In [ ]:
# In the data processing code we are doing the following
#1. Checking orginal dataset leakage
#2. Creating a leak free dataset by splitting the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Checking the data leakage issue of orginal dataset


import json
import pandas as pd

# -------------------------------
# 1. Load datasets
# -------------------------------
def load_json_dataset(path):
    with open(path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except:
            data = [json.loads(line) for line in f]
    return data

train_path = "/content/drive/MyDrive/Colab Notebooks/ObliQA_train.json"
eval_path  = "/content/drive/MyDrive/Colab Notebooks/ObliQA_dev.json"
test_path  = "/content/drive/MyDrive/Colab Notebooks/ObliQA_test.json"

data_train = load_json_dataset(train_path)
data_eval  = load_json_dataset(eval_path)
data_test  = load_json_dataset(test_path)

print("Train samples:", len(data_train))
print("Eval samples :", len(data_eval))
print("Test samples :", len(data_test))




# -------------------------------
# 2. Extract Document IDs
# -------------------------------
def get_doc_ids(dataset):
    """Extract unique DocumentIDs from either flattened or nested format."""
    if not dataset:
        return []
    # Flattened format: direct DocumentID field
    if "DocumentID" in dataset[0]:
        return sorted({str(entry["DocumentID"]) for entry in dataset})
    # Nested format: each question contains Passages list
    else:
        return sorted({str(p["DocumentID"]) for entry in dataset for p in entry.get("Passages", [])})

train_docs = get_doc_ids(data_train)
eval_docs  = get_doc_ids(data_eval)
test_docs  = get_doc_ids(data_test)



# -------------------------------
# 3. Print summary + overlaps
# -------------------------------
print("\nCounts:")
print(" Train Docs:", len(train_docs))
print(" Eval Docs :", len(eval_docs))
print(" Test Docs :", len(test_docs))

print("\nOverlaps (should all be 0):")
print(" Train ∩ Eval:", len(set(train_docs) & set(eval_docs)))
print(" Train ∩ Test:", len(set(train_docs) & set(test_docs)))
print(" Eval ∩ Test :", len(set(eval_docs) & set(test_docs)))



# -------------------------------
# 4. Table view
# -------------------------------
max_len = max(len(train_docs), len(eval_docs), len(test_docs))
df = pd.DataFrame({
    "Train DocumentIDs": train_docs + [""] * (max_len - len(train_docs)),
    "Eval DocumentIDs":  eval_docs  + [""] * (max_len - len(eval_docs)),
    "Test DocumentIDs":  test_docs  + [""] * (max_len - len(test_docs))
})

print("\nDocumentID Distribution Across Splits:")
print(df.to_string(index=False))


Train samples: 22295
Eval samples : 2788
Test samples : 2786

Counts:
 Train Docs: 40
 Eval Docs : 37
 Test Docs : 38

Overlaps (should all be 0):
 Train ∩ Eval: 37
 Train ∩ Test: 38
 Eval ∩ Test : 37

📊 DocumentID Distribution Across Splits:
Train DocumentIDs Eval DocumentIDs Test DocumentIDs
                1                1                1
               10               10               10
               11               11               11
               12               12               12
               13               13               13
               14               14               14
               15               15               15
               16               16               16
               17               17               17
               18               18               18
               19               19               19
                2                2                2
               20               21               21
               21            

In [ ]:
# Splitting the dataset for leak free dataset

import json, random
from collections import defaultdict

# --------------------------
# Step 1: Load original data
# --------------------------
train_path = "/content/drive/MyDrive/Colab Notebooks/ObliQA_train.json"
eval_path  = "/content/drive/MyDrive/Colab Notebooks/ObliQA_dev.json"
test_path  = "/content/drive/MyDrive/Colab Notebooks/ObliQA_test.json"

with open(train_path) as f: data_train = json.load(f)
with open(eval_path)  as f: data_eval  = json.load(f)
with open(test_path)  as f: data_test  = json.load(f)

full_data = data_train + data_eval + data_test
print(f"Total questions in raw ObliQA: {len(full_data)}")

# --------------------------
# Step 2: Split single vs multi-passage Qs
# --------------------------
single_pass = [q for q in full_data if len({p["DocumentID"] for p in q["Passages"]}) == 1]
multi_pass  = [q for q in full_data if len({p["DocumentID"] for p in q["Passages"]}) > 1]

print(f"Single-passage Qs: {len(single_pass)}")
print(f"Multi-passage Qs : {len(multi_pass)}")

# --------------------------
# Step 3: Group single-pass Qs by DocumentID
# --------------------------
doc_to_qs = defaultdict(list)
for q in single_pass:
    doc_id = q["Passages"][0]["DocumentID"]  # safe since only 1 doc
    doc_to_qs[doc_id].append(q)

docs = list(doc_to_qs.keys())
print(f"Unique documents (from single-pass Qs): {len(docs)}")

# --------------------------
# Step 4: Split documents into 80/10/10
# --------------------------
random.seed(42)  # reproducibility
random.shuffle(docs)

n = len(docs)
train_docs = set(docs[:int(0.8*n)])
eval_docs  = set(docs[int(0.8*n):int(0.95*n)])   # 15%
test_docs  = set(docs[int(0.95*n):])             # 5%


print(f"Train docs: {len(train_docs)}, Eval docs: {len(eval_docs)}, Test docs: {len(test_docs)}")

# --------------------------
# Step 5: Assign single-pass Qs to splits
# --------------------------
train_split, eval_split, test_split = [], [], []

for doc, qs in doc_to_qs.items():
    if doc in train_docs:
        train_split.extend(qs)
    elif doc in eval_docs:
        eval_split.extend(qs)
    elif doc in test_docs:
        test_split.extend(qs)

print(f"After single-pass split → Train: {len(train_split)}, Eval: {len(eval_split)}, Test: {len(test_split)}")

# --------------------------
# Step 6: Add multi-passage Qs if all docs in same split
# --------------------------
kept_multi, dropped_multi = 0, 0
sampled_examples = {"train": [], "eval": [], "test": [], "drop": []}

for q in multi_pass:
    doc_ids = {p["DocumentID"] for p in q["Passages"]}
    # try to extract text safely (different datasets use different keys)
    q_text = q.get("question") or q.get("Question") or q.get("question_text") or ""
    q_text = q_text[:100].replace("\n", " ")  # shorten for printing

    if doc_ids.issubset(train_docs):
        train_split.append(q)
        kept_multi += 1
        if len(sampled_examples["train"]) < 3:
            sampled_examples["train"].append((q_text, doc_ids))
    elif doc_ids.issubset(eval_docs):
        eval_split.append(q)
        kept_multi += 1
        if len(sampled_examples["eval"]) < 3:
            sampled_examples["eval"].append((q_text, doc_ids))
    elif doc_ids.issubset(test_docs):
        test_split.append(q)
        kept_multi += 1
        if len(sampled_examples["test"]) < 3:
            sampled_examples["test"].append((q_text, doc_ids))
    else:
        dropped_multi += 1
        if len(sampled_examples["drop"]) < 3:
            sampled_examples["drop"].append((q_text, doc_ids))

print(f"\nMulti-pass Qs kept: {kept_multi}, dropped: {dropped_multi}")
print(f"Final splits → Train: {len(train_split)}, Eval: {len(eval_split)}, Test: {len(test_split)}")




Total questions in raw ObliQA: 27869
Single-passage Qs: 25699
Multi-passage Qs : 2170
Unique documents (from single-pass Qs): 40
Train docs: 32, Eval docs: 6, Test docs: 2
After single-pass split → Train: 19411, Eval: 3178, Test: 3110

Multi-pass Qs kept: 1181, dropped: 989
Final splits → Train: 20573, Eval: 3191, Test: 3116


In [ ]:

# --------------------------
# Step 7: Save splits to Drive
# --------------------------
save_dir = "/content/drive/MyDrive/Colab Notebooks/Obliqa_new"
os.makedirs(save_dir, exist_ok=True)

train_out = os.path.join(save_dir, "train.json")
eval_out  = os.path.join(save_dir, "eval.json")
test_out  = os.path.join(save_dir, "test.json")

with open(train_out, "w") as f:
    json.dump(train_split, f, indent=2)

with open(eval_out, "w") as f:
    json.dump(eval_split, f, indent=2)

with open(test_out, "w") as f:
    json.dump(test_split, f, indent=2)

print(" Saved splits to Google Drive:")
print(f"   Train → {train_out} ({len(train_split)} questions)")
print(f"   Eval  → {eval_out} ({len(eval_split)} questions)")
print(f"   Test  → {test_out} ({len(test_split)} questions)")


✅ Saved splits to Google Drive:
   Train → /content/drive/MyDrive/Colab Notebooks/Obliqa_new/train.json (20573 questions)
   Eval  → /content/drive/MyDrive/Colab Notebooks/Obliqa_new/eval.json (3191 questions)
   Test  → /content/drive/MyDrive/Colab Notebooks/Obliqa_new/test.json (3116 questions)


In [ ]:

## Checking the new leak free distribution of documents

import json
import pandas as pd



def get_doc_ids(dataset):
    if not dataset:
        return []
    if "DocumentID" in dataset[0]:  # flattened format
        return sorted({str(entry["DocumentID"]) for entry in dataset})
    else:  # original ObliQA format
        return sorted({str(p["DocumentID"]) for entry in dataset for p in entry.get("Passages", [])})

train_docs = get_doc_ids(train_split)
eval_docs  = get_doc_ids(eval_split)
test_docs  = get_doc_ids(test_split)

print("Counts:")
print(" Train Docs:", len(train_docs))
print(" Eval Docs :", len(eval_docs))
print(" Test Docs :", len(test_docs))

print("\nOverlaps (should all be 0):")
print(" Train ∩ Eval:", len(set(train_docs) & set(eval_docs)))
print(" Train ∩ Test:", len(set(train_docs) & set(test_docs)))
print(" Eval ∩ Test :", len(set(eval_docs) & set(test_docs)))

# Build a side-by-side table
max_len = max(len(train_docs), len(eval_docs), len(test_docs))
df = pd.DataFrame({
    "Train DocumentIDs": train_docs + [""] * (max_len - len(train_docs)),
    "Eval DocumentIDs": eval_docs + [""] * (max_len - len(eval_docs)),
    "Test DocumentIDs": test_docs + [""] * (max_len - len(test_docs))
})

print("\n DocumentID distribution across splits:")
print(df.to_string(index=False))

Counts:
 Train Docs: 32
 Eval Docs : 6
 Test Docs : 2

Overlaps (should all be 0):
 Train ∩ Eval: 0
 Train ∩ Test: 0
 Eval ∩ Test : 0

 DocumentID distribution across splits:
Train DocumentIDs Eval DocumentIDs Test DocumentIDs
                1               10                3
               11               14                7
               12                2                 
               13               21                 
               15               27                 
               16               30                 
               17                                  
               18                                  
               19                                  
               20                                  
               22                                  
               23                                  
               24                                  
               25                                  
               26                            

In [ ]:
#%pip install rapidfuzz sentence-transformers

In [ ]:
# Now checking the Overlaps and Semantic similarity between train, eval and test dataset

import json
from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer, util

# --------------------------
# Step 1: Extract questions
# --------------------------
def extract_questions(split):
    return [ (q.get("question") or q.get("Question") or q.get("question_text") or "").strip()
             for q in split if (q.get("question") or q.get("Question") or q.get("question_text")) ]

train_qs = extract_questions(train_split)
eval_qs  = extract_questions(eval_split)

print(f"Train questions: {len(train_qs)}")
print(f"Eval questions : {len(eval_qs)}")

# --------------------------
# Step 2: Exact duplicates
# --------------------------
train_set = set([q.lower() for q in train_qs])
eval_set  = set([q.lower() for q in eval_qs])

exact_dupes = train_set.intersection(eval_set)
print(f"\n🔎 Exact duplicates between train & eval: {len(exact_dupes)}")
for i, q in enumerate(list(exact_dupes)[:10]):
    print(f"  {i+1}. {q}")

# --------------------------
# Step 3: Near-duplicates (fuzzy)
# --------------------------
similar_pairs = []
for eq in eval_qs:
    best_match, score, _ = process.extractOne(eq, train_qs, scorer=fuzz.token_sort_ratio)
    if score >= 85:  # threshold = 85
        similar_pairs.append((eq, best_match, score))

print(f"\n🔎 Near-duplicate overlaps (fuzzy ≥ 85): {len(similar_pairs)}")
for i, (q_eval, q_train, score) in enumerate(similar_pairs[:5]):
    print(f"\nEval: {q_eval}\nTrain: {q_train}\nScore: {score}")

# --------------------------
# Step 4: Semantic similarity (embeddings)
# --------------------------
print("\n🔎 Running semantic similarity check with embeddings... (this may take time)")

# load model (small & fast)
model = SentenceTransformer("all-MiniLM-L6-v2")

# encode
train_emb = model.encode(train_qs, convert_to_tensor=True, show_progress_bar=True)
eval_emb  = model.encode(eval_qs, convert_to_tensor=True, show_progress_bar=True)

# compute cosine similarities
semantic_overlaps = []
threshold = 0.85

for i, e_emb in enumerate(eval_emb):
    sims = util.cos_sim(e_emb, train_emb)[0]
    max_sim, idx = sims.max().item(), sims.argmax().item()
    if max_sim >= threshold:
        semantic_overlaps.append((eval_qs[i], train_qs[idx], max_sim))

print(f"\n🔎 Semantic overlaps (cosine ≥ {threshold}): {len(semantic_overlaps)}")
for i, (q_eval, q_train, score) in enumerate(semantic_overlaps[:5]):
    print(f"\nEval: {q_eval}\nTrain: {q_train}\nScore: {score:.3f}")


Train questions: 20573
Eval questions : 3191

🔎 Exact duplicates between train & eval: 0

🔎 Near-duplicate overlaps (fuzzy ≥ 85): 4

Eval: How frequently does the Regulator conduct periodic risk assessments for Recognised Bodies, and what triggers might prompt an out-of-cycle review?
Train: How frequently does the Regulator conduct periodic risk assessments for Recognised Bodies, and what triggers might prompt an out-of-cycle risk assessment?
Score: 93.64548494983278

Eval: What constitutes a "proven track record of reasonable accuracy" for an Authorised Person's models in measuring risk, and how is this evaluated?
Train: What constitutes a "proven track record of reasonable accuracy" in measuring risk for an Authorised Person's models, and how is this evaluated by the Regulator?
Score: 93.06930693069306

Eval: Is there a prescribed format or template that the ADGM recommends or requires for the disclosure of Exploration Results to ensure consistency and compliance with Rule 11.4?
Trai

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/643 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]


🔎 Semantic overlaps (cosine ≥ 0.85): 139

Eval: What rule would the Regulator invoke if it finds that a Recognised Body has not complied with a specific regulatory direction?
Train: Under what conditions might a Regulator issue a Direction to a Recognised Body regarding its compliance with the Rules?
Score: 0.909

Eval: Is there a specified or preferred framework or set of best practices that the ADGM recommends for effectively measuring, monitoring, and managing liquidity risk in accordance with Rule 4.7.23?
Train: What specific metrics or indicators are recommended by the ADGM for measuring and monitoring Liquidity Risk to ensure they align with the requirements set out in Rule 9.2.5?
Score: 0.865

Eval: What are the ADGM's expectations for stress and scenario testing in liquidity risk management, and how should these tests be documented and reviewed?
Train: What are the ADGM's expectations regarding the stress testing scenarios that should be included when managing Liquidity Risk?
